## 1. Entraînement

In [ ]:
from datasets import load_dataset

data_files = {
    "train": "train.jsonl",
    "validation": "valid.jsonl",
}
ds = load_dataset("json", data_files=data_files)

print(ds)
print(ds["train"][0].keys())

In [ ]:
import math
import mlflow
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from sentence_transformers.losses.TripletLoss import TripletDistanceMetric
from torch.utils.data import DataLoader

# --- CONFIGURATION ---
BASE_MODEL = "intfloat/multilingual-e5-base"
OUT_DIR = "Fe2x/finetuned-embedder"
EXP_NAME = "E5_Triplet_Finetuning"
PARAMS = {
    "epochs": 1,
    "batch_size": 16,
    "triplet_margin": 0.2,
    "lr": 2e-5,
    "warmup_ratio": 0.1
}

# 1. Préparation des données (Format E5)
def to_input_examples(split):
    return [
        InputExample(
            texts=[
                "query: " + r["query"],
                "passage: " + r["chunk_pos"],
                "passage: " + r["chunk_neg"],
            ]
        )
        for r in split
    ]

# Supposons que ds["train"] et ds["test"] existent
train_examples = to_input_examples(ds["train"])
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=PARAMS["batch_size"])

# Création d'un évaluateur pour avoir des métriques dans MLflow
val_examples = to_input_examples(ds["test"]) # Utilise un subset si c'est trop long
evaluator = evaluation.TripletEvaluator.from_input_examples(
    val_examples, 
    name='eval_triplet',
    distance_metric=TripletDistanceMetric.COSINE
)

# 2. Initialisation du modèle et de la Loss
model = SentenceTransformer(BASE_MODEL)
loss = losses.TripletLoss(
    model=model,
    distance_metric=TripletDistanceMetric.COSINE,
    triplet_margin=PARAMS["triplet_margin"],
)

# 3. Entraînement avec MLflow
mlflow.set_experiment(EXP_NAME)

with mlflow.start_run():
    # Log des hyperparamètres
    mlflow.log_params(PARAMS)
    mlflow.log_param("base_model", BASE_MODEL)

    warmup_steps = math.ceil(len(train_dataloader) * PARAMS["epochs"] * PARAMS["warmup_ratio"])

    # Callback pour envoyer les métriques à MLflow à chaque évaluation
    def mlflow_callback(score, epoch, steps):
        mlflow.log_metric("triplet_accuracy", score, step=steps)

    model.fit(
        train_objectives=[(train_dataloader, loss)],
        epochs=PARAMS["epochs"],
        warmup_steps=warmup_steps,
        output_path=OUT_DIR,
        evaluator=evaluator,
        evaluation_steps=100, # Ajuste selon la taille de tes données
        callback=mlflow_callback,
        show_progress_bar=True,
    )

    # Sauvegarde du modèle final en tant qu'artéfact MLflow
    mlflow.log_artifacts(OUT_DIR, artifact_path="model_files")
    
    print(f"✅ Training finished. Model and logs saved.")

In [ ]:
import os

model.push_to_hub(
    "Fe2x/multilingual-e5-base-instruct",
    private=True,
)

## 2. Test & Validation

### 2.1 Accuracy

In [ ]:
from sentence_transformers import SentenceTransformer

# MODEL_ID = "Fe2x/finetuned-embedding-rag"
# model = SentenceTransformer(OUT_DIR)
base = SentenceTransformer("intfloat/multilingual-e5-base")

import pandas as pd

valid = pd.read_json("valid.jsonl", lines=True)
valid.head()

In [ ]:
print("Base acc:", triplet_accuracy(base, valid))
print("FT   acc:", triplet_accuracy(model, valid))

In [ ]:
import numpy as np

def triplet_accuracy(model, df):
    ok = 0
    for _, r in df.iterrows():
        q = model.encode(r["query"], normalize_embeddings=True)
        p = model.encode(r["chunk_pos"], normalize_embeddings=True)
        n = model.encode(r["chunk_neg"], normalize_embeddings=True)

        if float(np.dot(q, p)) > float(np.dot(q, n)):
            ok += 1
    return ok / len(df)

acc = triplet_accuracy(model, valid)
print(f"Triplet accuracy (valid): {acc:.3f}")


### 2.2 Autres metriques à enregistrer sur mlflow

In [ ]:
import json

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


valid = load_jsonl("valid.jsonl")

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers import SentenceTransformer
import numpy as np
import mlflow

def build_full_corpus_ir(triplets):
    queries = {}
    relevant_docs = {}
    corpus = {}

    # corpus = tous les chunks positifs uniques
    for t in triplets:
        pid = str(t["pos_id"])
        corpus[pid] = t["chunk_pos"]

    # queries + relevances
    for i, t in enumerate(triplets):
        qid = str(i)
        queries[qid] = t["query"]
        relevant_docs[qid] = {str(t["pos_id"])}

    return queries, corpus, relevant_docs


queries, corpus, relevant_docs = build_full_corpus_ir(valid)

evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    show_progress_bar=True,
    name="valid_ir",
)

# On récupère le run actif ou on en crée un nouveau si besoin
with mlflow.start_run(run_id=mlflow.last_active_run().info.run_id):
    
    print("Évaluation du modèle de base...")
    base_scores = evaluator(base)
    mlflow.log_metrics({f"base_{k}": v for k, v in base_scores.items() if isinstance(v, (int, float))})

    print("Évaluation du modèle Fine-tuné...")
    ft_scores = evaluator(model)
    mlflow.log_metrics({f"ft_{k}": v for k, v in ft_scores.items() if isinstance(v, (int, float))})

    print("-" * 20)
    print(f"Base NDCG@10: {base_scores.get('valid_ir_cosine_ndcg@10', 'N/A')}")
    print(f"FT NDCG@10:   {ft_scores.get('valid_ir_cosine_ndcg@10', 'N/A')}")





In [ ]:
q = "Comment installer des bibliothèques Python ?"
p = "Pour installer une bibliothèque Python, utilisez pip install ..."

e_base_q = base.encode(["query: " + q], convert_to_numpy=True)
e_ft_q   = model.encode(["query: " + q], convert_to_numpy=True)

e_base_p = base.encode(["passage: " + p], convert_to_numpy=True)
e_ft_p   = model.encode(["passage: " + p], convert_to_numpy=True)

import numpy as np
def cos(a,b):
    return float(np.dot(a[0], b[0]) / (np.linalg.norm(a[0])*np.linalg.norm(b[0]) + 1e-12))

print("cos(base_q, ft_q) =", cos(e_base_q, e_ft_q))
print("cos(base_p, ft_p) =", cos(e_base_p, e_ft_p))

